# Очистка таблиц Users, Items и Users_Items

В этом ноутбуке я привожу таблицы в нормальный вид, оставляю только нужные столбцы и строки.

## Импорт библиотек

In [368]:
import pandas as pd
import numpy as np
import os
import re
import scipy.stats as sts
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
from pathlib import Path

## Загрузка данных

In [369]:
BASE_PATH = os.path.abspath("../../../Tables/BaseTable")

In [370]:
users_df = pd.read_csv(os.path.join(BASE_PATH, 'Users.csv'), sep=';', encoding='cp1251')
items_df = pd.read_csv(os.path.join(BASE_PATH, 'Items.csv'), sep=';', encoding='cp1251')
user_item_df = pd.read_csv(os.path.join(BASE_PATH, 'User_Items_test_month.csv'), sep=';', encoding='cp1251')
user_item_year_df = pd.read_csv(os.path.join(BASE_PATH, 'User_items_year.csv'), sep=',', encoding='utf-8')

C:\Users\msgt0\AppData\Local\Temp\ipykernel_26464\3844528746.py:4: DtypeWarning: Columns (0: 	Телефон ) have mixed types. Specify dtype option on import or set low_memory=False.
  user_item_year_df = pd.read_csv(os.path.join(BASE_PATH, 'User_items_year.csv'), sep=',', encoding='utf-8')


In [371]:
print("Изначальные размеры:")
print(f"Users: {users_df.shape}")
print(f"Items: {items_df.shape}")
print(f"User_Items: {user_item_df.shape}")
print(f"User_Item_year: {user_item_year_df.shape}")

Изначальные размеры:
Users: (2893, 18)
Items: (258, 8)
User_Items: (3133, 17)
User_Item_year: (39289, 19)


In [372]:
users_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2893 entries, 0 to 2892
Data columns (total 18 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Имя                                        2892 non-null   str    
 1   Телефон                                    2892 non-null   str    
 2   Email                                      997 non-null    str    
 3   Категории                                  58 non-null     str    
 4   Дата рождения                              2 non-null      str    
 5   Потратил, ?                                2892 non-null   str    
 6   Оплатил, ?                                 2892 non-null   str    
 7   Пол                                        6 non-null      str    
 8   Карта                                      0 non-null      float64
 9   Скидка                                     2892 non-null   float64
 10  Последний визит                    

In [373]:
items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 258 entries, 0 to 257
Data columns (total 8 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   Название услуги или товара             258 non-null    str  
 1   Категория товара или услуги в продаже  258 non-null    str  
 2   Количество оказанных услуг             258 non-null    int64
 3   Доля от оказанных услуг в %            258 non-null    str  
 4   Выручка от продажи услуг, ?            258 non-null    str  
 5   % от общей выручки за услуги           258 non-null    str  
 6   Ср. выручка с одного клиента, ?        258 non-null    str  
 7   Уникальные клиенты                     258 non-null    int64
dtypes: int64(2), str(6)
memory usage: 16.3 KB


In [374]:
user_item_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3133 entries, 0 to 3132
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя	 мастера                    1777 non-null   str    
 1   Специализация мастера           1777 non-null   str    
 2   Имя клиента                     1154 non-null   str    
 3   	Телефон                        1154 non-null   float64
 4   Email                           502 non-null    str    
 5   Время визита                    1777 non-null   str    
 6   Имя	 создателя                  1777 non-null   str    
 7   Дата создания                   1777 non-null   str    
 8   Статус                          1777 non-null   str    
 9   Источник                        1777 non-null   str    
 10  Категория услуги                3096 non-null   str    
 11  Название услуги                 3096 non-null   str    
 12  Стоимость, руб                  3096 non-null

In [375]:
user_item_year_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39289 entries, 0 to 39288
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя	 мастера                    22726 non-null  str    
 1   Специализация мастера           22726 non-null  str    
 2   Имя клиента                     12835 non-null  str    
 3   	Телефон                        12835 non-null  object 
 4   Email                           5604 non-null   str    
 5   Время визита                    22726 non-null  str    
 6   Имя	 создателя                  22726 non-null  str    
 7   Дата создания                   22726 non-null  str    
 8   Статус                          22726 non-null  str    
 9   Источник                        22726 non-null  str    
 10  Категория услуги                39075 non-null  str    
 11  Название услуги                 39075 non-null  str    
 12  Стоимость, руб                  39075 non-n

Заранее изучив данные, выявлены необходимые операции:

__Для Users__

1) Вместо Имя+Телефон+Email ввести `id_user`, а Email удалить
2) Удалить столбцы Дата рождения, Потратил в руб; Карта; Скидка; Последний визит; Чаевые; Количество посещений; Комментарий; Дополнительный телефон; Согласен на получение рассылок; Согласен на обработку персональных данных т.к. они большинством своим не информативны либо полностью пустые
3) В столбце Категория есть несколько возможных значений я хочу:
- все пропуски и значения "Клиенты отщепенцев", "ПРЕДОПЛАТА" заменить на "обычный"
- удалить все экземпляры (строчки), где категория "Черный список"
- заменить значение "Постоянный" на "обычный" (т.к. в таблице только 1 такой экземпляр)
4) Удалить пропуски, если они останутся

__Для Items__

1) Удалить все услуги, где значение "Категория товара или услуги в продаже" = "Солярий" (слишком много посещений и направление никак не связано с остальными услугами)
2) Удалить все услуги, которые были оказаны слишком маленькое количество раз ("Количество оказанных услуг"<8 раз) или где количество уникальных клиентов слишком маленькое ("Уникальные клиенты" < 7).
3) По сути каждая строчка это и есть информация об уникальной услуге, но я бы добавил столбец `id_item` и сделал его PK

__Для User-Items__

1) Удалить все элементы связанные с солярием
2) Заменить имена столбцов на отношение сущьностей (то есть столбец "Имя", который относиться к клиенту назвать "Имя клиента" и т.п.)
3) Удалить Создал (Имя, Дата);Статус; Источник; Информация (Статус, Имя, Дата); Оплачено полностью; Комментарий; Категория (они не имеют смысла или нулевые)
4) Добавить `user_id` (для каждого клиента такой же как в таблице Users(то есть чтобы одному клиенту и там и там значился 1 и тот же id)), добавить `item_id` (по аналогии с `user_id`) и удалить "Email"
5) Объеденить некоторые услуги исходя из бизнесс логики 

__Для User-Items-Year__

1) Повторить pipline для User-Items, но на файле с годом  


## Обработка Users

#### Вводим user_id для каждого уникального пользователя, определяя его с помощью тройки Имя+телефон+Email

In [376]:
def clean_phone(phone):
    if pd.isna(phone):
        return np.nan
    phone_str = str(phone).replace('.0', '').replace('+7', '8')
    return phone_str.strip()

In [377]:
users_df["Телефон"] = users_df["Телефон"].apply(clean_phone)
users_df = users_df.drop_duplicates(subset=["Телефон"], keep="first")
users_df["id_user"] = pd.Categorical(users_df["Телефон"]).codes

#### Настройка столбца 'Категории'

In [378]:
users_df['Категории'] = users_df['Категории'].fillna('обычный')
users_df['Категории'] = users_df['Категории'].replace({
    'Клиенты отщепенцев': 'обычный',
    'ПРЕДОПЛАТА': 'обычный',
    'Постоянный': 'обычный'
})

In [379]:
users_df['Категории'].value_counts()

Категории
обычный          2881
Черный список      12
Name: count, dtype: int64

In [380]:
users_df = users_df[users_df['Категории'] != 'Черный список'].copy()
users_df['Категории'].value_counts()

Категории
обычный    2881
Name: count, dtype: int64

Оставляем столбец, т.к. в будущем возможно введение других категорий.

In [381]:
users_df = users_df[[
    'id_user', 'Имя', 'Телефон',
    'Категории', 'Оплатил, ?', 'Последний визит'
]].copy()
users_df = users_df.rename(columns={'Оплатил, ?': 'Оплачено в руб'})

In [382]:
users_df.info()

<class 'pandas.DataFrame'>
Index: 2881 entries, 0 to 2892
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id_user          2881 non-null   int16
 1   Имя              2880 non-null   str  
 2   Телефон          2880 non-null   str  
 3   Категории        2881 non-null   str  
 4   Оплачено в руб   2880 non-null   str  
 5   Последний визит  2711 non-null   str  
dtypes: int16(1), str(5)
memory usage: 140.7 KB


In [383]:
users_df.head()

,id_user,Имя,Телефон,Категории,Оплачено в руб,Последний визит
0,2706,Наталья,79859657657,обычный,50363,2026-03-04 13:00
1,462,дарья,79082451054,обычный,0,2026-03-04 13:00
2,965,Татьяна,79163219575,обычный,30880,2026-03-04 12:30
3,1924,Анастасия,79279021487,обычный,41470,2026-03-04 12:05
4,1206,Екатерина,79168230729,обычный,171450,2026-03-04 12:00


In [384]:
users_df[users_df['Телефон']=='79165161952']

,id_user,Имя,Телефон,Категории,Оплачено в руб,Последний визит
121,1050,Арина Фомичева,79165161952,обычный,58040,2026-02-28 14:05


#### Удалим пропуски и зафиксируем таблицу

In [385]:
users_clean = users_df.dropna(subset=['Последний визит']).copy()

In [386]:
users_clean.isna().sum()

id_user            0
Имя                0
Телефон            0
Категории          0
Оплачено в руб     0
Последний визит    0
dtype: int64

In [387]:
users_clean.info()

<class 'pandas.DataFrame'>
Index: 2711 entries, 0 to 2719
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id_user          2711 non-null   int16
 1   Имя              2711 non-null   str  
 2   Телефон          2711 non-null   str  
 3   Категории        2711 non-null   str  
 4   Оплачено в руб   2711 non-null   str  
 5   Последний визит  2711 non-null   str  
dtypes: int16(1), str(5)
memory usage: 132.4 KB


#### Настроем типы данных

In [388]:
users_clean['Оплачено в руб'] = users_clean['Оплачено в руб'].astype(str).str.replace(',', '.').astype(float)

In [389]:
users_clean['Последний визит'] = pd.to_datetime(users_clean['Последний визит'], errors='coerce')

In [390]:
users_clean.dtypes

id_user                     int16
Имя                           str
Телефон                       str
Категории                     str
Оплачено в руб            float64
Последний визит    datetime64[us]
dtype: object

## Обработка Items

In [391]:
items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 258 entries, 0 to 257
Data columns (total 8 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   Название услуги или товара             258 non-null    str  
 1   Категория товара или услуги в продаже  258 non-null    str  
 2   Количество оказанных услуг             258 non-null    int64
 3   Доля от оказанных услуг в %            258 non-null    str  
 4   Выручка от продажи услуг, ?            258 non-null    str  
 5   % от общей выручки за услуги           258 non-null    str  
 6   Ср. выручка с одного клиента, ?        258 non-null    str  
 7   Уникальные клиенты                     258 non-null    int64
dtypes: int64(2), str(6)
memory usage: 16.3 KB


# Новая предобработка

#### Переименовываем столбцы 

In [392]:
items_df.columns = items_df.columns.str.strip()  # убираем пробелы в именах
items_df = items_df.rename(columns={
    "Выручка от продажи услуг, ?": "Выручка от продажи услуг, руб",
    "Ср. выручка с одного клиента, ?": "Ср. выручка с одного клиента, руб"
})

#### Настраиваем типизацию

In [393]:
def clean_money_col(series):
    return pd.to_numeric(series.str.replace(" ", "").str.replace(",", "."), errors="coerce")

def clean_percent_col(series):
    # убираем %, пробелы, заменяем запятые, приводим к числу
    return pd.to_numeric(
        series.str.replace(" ", "")
           .str.replace("%", "")
           .str.replace(",", "."),
        errors="coerce"
    )

items_df["Количество оказанных услуг"] = pd.to_numeric(
    items_df["Количество оказанных услуг"], errors="coerce"
).astype("Int64")

items_df["Доля от оказанных услуг в %"] = clean_percent_col(items_df["Доля от оказанных услуг в %"])
items_df["Выручка от продажи услуг, руб"] = clean_money_col(items_df["Выручка от продажи услуг, руб"])
items_df["% от общей выручки за услуги"] = clean_percent_col(items_df["% от общей выручки за услуги"])
items_df["Ср. выручка с одного клиента, руб"] = clean_money_col(items_df["Ср. выручка с одного клиента, руб"])
items_df["Уникальные клиенты"] = pd.to_numeric(
    items_df["Уникальные клиенты"], errors="coerce"
).astype("Int64")

In [394]:
items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 258 entries, 0 to 257
Data columns (total 8 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Название услуги или товара             258 non-null    str    
 1   Категория товара или услуги в продаже  258 non-null    str    
 2   Количество оказанных услуг             258 non-null    Int64  
 3   Доля от оказанных услуг в %            258 non-null    float64
 4   Выручка от продажи услуг, руб          258 non-null    float64
 5   % от общей выручки за услуги           258 non-null    float64
 6   Ср. выручка с одного клиента, руб      258 non-null    float64
 7   Уникальные клиенты                     258 non-null    Int64  
dtypes: Int64(2), float64(4), str(2)
memory usage: 16.8 KB


Удалчем услуги с солярием 

In [395]:
items_no_solar = items_df[
    items_df["Категория товара или услуги в продаже"] != "Солярий"
].copy()

In [396]:
items_filtered = items_no_solar[
    (items_no_solar["Количество оказанных услуг"] >= 6) &
    (items_no_solar["Уникальные клиенты"] >= 5)
].copy()

## Создаем items_dim

In [397]:
rename_map = {
    # брови
    "коррекция бровей/форма бровей (жен.)": "Коррекция бровей/форма бровей",
    "коррекция бровей/форма бровей (муж.)": "Коррекция бровей/форма бровей",

    # укрепление
    "укрепление акрилом": "Укрепление",
    "укрепление гелем": "Укрепление",

    # покрытие
    "укрепление цветным гелем": "Покрытие",
    "покрытие гель лаком luxio, emi, onenail (руки)": "Покрытие",
    "покрытие стойким лаком (ноги)": "Покрытие",
    "покрытие база + топ (бесцветные)": "Покрытие",
    "покрытие гель лаком со светоотражающим эффектом (руки)": "Покрытие",
    "покрытие стойким лаком (руки)": "Покрытие",
    "покрытие гель лаком luxio, emi, onenail (ноги)": "Покрытие",
    "покрытие гель лаком emi, onenail (руки)": "Покрытие",
    "покрытие гель лаком со светоотражающим эффектом (ноги)": "Покрытие",
    "покрытие гель лаком luxio, beautix, e.mi, onenail (руки) 1 ноготь": "Покрытие",
    "покрытие гель лаком с эффектом \"кошачий глаз\" (руки)": "Покрытие",
    "покрытие гель лаком luxio, beautix, onenail френч (руки)": "Покрытие",

    # снятие
    "снятие лака (руки)": "Снятие",
    "снятие гель-лака (1 ноготь)": "Снятие",
    "снятие гель-лака": "Снятие",
    "снятие стойкого лака (руки)": "Снятие",
    "снятие лака (ноги)": "Снятие",
    "снятие укрепления гелем / нарощенных ногтей (10 ногтей)": "Снятие",
}

In [398]:
ITEM_COL = "Название услуги или товара"

items_df[ITEM_COL] = items_df[ITEM_COL].str.strip().str.lower().replace(rename_map)

In [399]:
items_df = items_df.groupby(ITEM_COL).agg({
    'Категория товара или услуги в продаже': 'first', # Берем первую попавшуюся категорию
    'Количество оказанных услуг': 'sum',            # Суммируем количество
    'Выручка от продажи услуг, руб': 'sum',         # Суммируем выручку
    'Уникальные клиенты': 'sum',                    # здесь может быть погрешность (один клиент мог брать обе услуги)
    # Для долей и процентов лучше пересчитать их заново после группировки, 
    # либо взять среднее, если это допустимо:
    'Доля от оказанных услуг в %': 'mean',
    '% от общей выручки за услуги': 'mean',
    'Ср. выручка с одного клиента, руб': 'mean'
}).reset_index()

#### Добавим id_item

In [400]:
items_dim_cols = [
    "Название услуги или товара",
    "Категория товара или услуги в продаже"
]

# присвоим id_item (сквозной integer, как в твоей схеме)
items_dim = (items_df[items_dim_cols]
    .reset_index(drop=True)
    .reset_index()  # index = 0..N-1
    .rename(columns={"index": "id_item"})
)

## Создание items_stats

In [401]:
items_stats_cols = [
    "id_item",
    "Название услуги или товара",
    "Категория товара или услуги в продаже",
    "Количество оказанных услуг",
    "Доля от оказанных услуг в %",
    "Выручка от продажи услуг, руб",
    "% от общей выручки за услуги",
    "Ср. выручка с одного клиента, руб",
    "Уникальные клиенты",
]

In [402]:
items_df["id_item"] = range(1, len(items_df) + 1)

items_stats = items_df[items_stats_cols].copy()

Упорядочиваем

In [403]:
items_stats = items_stats.sort_values("id_item").reset_index(drop=True)

In [404]:
items_stats.info()

<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 9 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   id_item                                240 non-null    int64  
 1   Название услуги или товара             240 non-null    str    
 2   Категория товара или услуги в продаже  240 non-null    str    
 3   Количество оказанных услуг             240 non-null    Int64  
 4   Доля от оказанных услуг в %            240 non-null    float64
 5   Выручка от продажи услуг, руб          240 non-null    float64
 6   % от общей выручки за услуги           240 non-null    float64
 7   Ср. выручка с одного клиента, руб      240 non-null    float64
 8   Уникальные клиенты                     240 non-null    Int64  
dtypes: Int64(2), float64(4), int64(1), str(2)
memory usage: 17.5 KB


## Обработка User-Items-Year

Удаляем пустые столбцы 

In [405]:
user_item_year_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39289 entries, 0 to 39288
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя	 мастера                    22726 non-null  str    
 1   Специализация мастера           22726 non-null  str    
 2   Имя клиента                     12835 non-null  str    
 3   	Телефон                        12835 non-null  object 
 4   Email                           5604 non-null   str    
 5   Время визита                    22726 non-null  str    
 6   Имя	 создателя                  22726 non-null  str    
 7   Дата создания                   22726 non-null  str    
 8   Статус                          22726 non-null  str    
 9   Источник                        22726 non-null  str    
 10  Категория услуги                39075 non-null  str    
 11  Название услуги                 39075 non-null  str    
 12  Стоимость, руб                  39075 non-n

In [406]:
user_item_year_df = user_item_year_df.loc[:, ~user_item_year_df.columns.str.contains("^Unnamed")]

Почистить имена колонок 

In [407]:
user_item_year_df.columns = user_item_year_df.columns.str.strip().str.replace(r'\t','', regex=True)

In [408]:
user_item_year_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39289 entries, 0 to 39288
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя мастера                     22726 non-null  str    
 1   Специализация мастера           22726 non-null  str    
 2   Имя клиента                     12835 non-null  str    
 3   Телефон                         12835 non-null  object 
 4   Email                           5604 non-null   str    
 5   Время визита                    22726 non-null  str    
 6   Имя создателя                   22726 non-null  str    
 7   Дата создания                   22726 non-null  str    
 8   Статус                          22726 non-null  str    
 9   Источник                        22726 non-null  str    
 10  Категория услуги                39075 non-null  str    
 11  Название услуги                 39075 non-null  str    
 12  Стоимость, руб                  39075 non-n

In [409]:
COL_MASTER = "Имя мастера"
COL_MASTER_SPEC = "Специализация мастера"
COL_CLIENT_NAME = "Имя клиента"
COL_PHONE = "Телефон"
COL_EMAIL = "Email"
COL_VISIT_TIME = "Время визита"
COL_CATEGORY = "Категория услуги"
COL_SERVICE = "Название услуги"
COL_PRICE_DISCOUNT = "Стоимость с учетом скидки, руб"

In [410]:
def clean_money(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str)
              .str.replace(" ", "", regex=False)
              .str.replace(",", ".", regex=False),
        errors="coerce"
    )

Приведение типов данных 

In [411]:
user_item_year_df[COL_PRICE_DISCOUNT] = clean_money(user_item_year_df[COL_PRICE_DISCOUNT])

Удалим строки без услуг (пропуски)

In [412]:
user_item_year_df = user_item_year_df[user_item_year_df[COL_SERVICE].notna()]
user_item_year_df = user_item_year_df[user_item_year_df[COL_SERVICE].str.strip() != ""]

In [413]:
user_item_year_df.shape[0]

39075

Фильтрация по солярию и абонимаентам 

In [414]:
mask_not_solar = user_item_year_df[COL_CATEGORY] != "Солярий"
mask_not_sub = user_item_year_df[COL_CATEGORY] != "Услуги ПО АБОНЕМЕНТАМ"
user_item_year_df = user_item_year_df[mask_not_solar & mask_not_sub].copy()

In [415]:
user_item_year_df.shape[0]

28857

In [416]:
user_item_year_df.info()

<class 'pandas.DataFrame'>
Index: 28857 entries, 0 to 39286
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя мастера                     12436 non-null  str    
 1   Специализация мастера           12436 non-null  str    
 2   Имя клиента                     12379 non-null  str    
 3   Телефон                         12379 non-null  object 
 4   Email                           5490 non-null   str    
 5   Время визита                    12436 non-null  str    
 6   Имя создателя                   12436 non-null  str    
 7   Дата создания                   12436 non-null  str    
 8   Статус                          12436 non-null  str    
 9   Источник                        12436 non-null  str    
 10  Категория услуги                28857 non-null  str    
 11  Название услуги                 28857 non-null  str    
 12  Стоимость, руб                  28857 non-null  

Заполнение пропусков (полупстых строк)

In [417]:
user_item_year_df = user_item_year_df.sort_values([COL_VISIT_TIME, COL_CLIENT_NAME], ascending=False)

In [418]:
fill_cols = [col for col in user_item_year_df.columns if col != COL_SERVICE]
user_item_year_df[fill_cols] = user_item_year_df[fill_cols].ffill(axis=0)

In [419]:
user_item_year_df.info()

<class 'pandas.DataFrame'>
Index: 28857 entries, 30531 to 39286
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя мастера                     28857 non-null  str    
 1   Специализация мастера           28857 non-null  str    
 2   Имя клиента                     28857 non-null  str    
 3   Телефон                         28857 non-null  object 
 4   Email                           28855 non-null  str    
 5   Время визита                    28857 non-null  str    
 6   Имя создателя                   28857 non-null  str    
 7   Дата создания                   28857 non-null  str    
 8   Статус                          28857 non-null  str    
 9   Источник                        28857 non-null  str    
 10  Категория услуги                28857 non-null  str    
 11  Название услуги                 28857 non-null  str    
 12  Стоимость, руб                  28857 non-nu

Ренейм (объединение item)

In [420]:
rename_map = {
    # брови
    "коррекция бровей/форма бровей (жен.)": "Коррекция бровей/форма бровей",
    "коррекция бровей/форма бровей (муж.)": "Коррекция бровей/форма бровей",

    # укрепление
    "укрепление акрилом": "Укрепление",
    "укрепление гелем": "Укрепление",

    # покрытие
    "укрепление цветным гелем": "Покрытие",
    "покрытие гель лаком luxio, emi, onenail (руки)": "Покрытие",
    "покрытие стойким лаком (ноги)": "Покрытие",
    "покрытие база + топ (бесцветные)": "Покрытие",
    "покрытие гель лаком со светоотражающим эффектом (руки)": "Покрытие",
    "покрытие стойким лаком (руки)": "Покрытие",
    "покрытие гель лаком luxio, emi, onenail (ноги)": "Покрытие",
    "покрытие гель лаком emi, onenail (руки)": "Покрытие",
    "покрытие гель лаком со светоотражающим эффектом (ноги)": "Покрытие",
    "покрытие гель лаком luxio, beautix, e.mi, onenail (руки) 1 ноготь": "Покрытие",
    "покрытие гель лаком с эффектом \"кошачий глаз\" (руки)": "Покрытие",
    "покрытие гель лаком luxio, beautix, onenail френч (руки)": "Покрытие",

    # снятие
    "снятие лака (руки)": "Снятие",
    "снятие гель-лака (1 ноготь)": "Снятие",
    "снятие гель-лака": "Снятие",
    "снятие стойкого лака (руки)": "Снятие",
    "снятие лака (ноги)": "Снятие",
    "снятие укрепления гелем / нарощенных ногтей (10 ногтей)": "Снятие",
}

In [421]:
user_item_year_df[COL_SERVICE] = user_item_year_df[COL_SERVICE].str.strip().str.lower().replace(rename_map)

In [422]:
user_item_year_df.shape

(28857, 17)

Добавляем id_user

In [423]:
users_df["Телефон"] = users_df["Телефон"].astype(str).str.strip()
user_item_year_df[COL_PHONE] = user_item_year_df[COL_PHONE].astype(str).str.strip()

In [424]:
user_item_year_df["Телефон"] = user_item_year_df["Телефон"].apply(clean_phone)

In [425]:
user_item_year_df = user_item_year_df.merge(
    users_df[["id_user", "Телефон"]],
    on="Телефон",
    how="left"
)

In [426]:
user_item_year_df.isna().sum()

Имя мастера                        0
Специализация мастера              0
Имя клиента                        0
Телефон                            0
Email                              2
Время визита                       0
Имя создателя                      0
Дата создания                      0
Статус                             0
Источник                           0
Категория услуги                   0
Название услуги                    0
Стоимость, руб                     0
Стоимость с учетом скидки, руб     0
Оплачено полностью                 0
Комментарий                        6
Категория                         20
id_user                           31
dtype: int64

In [427]:
user_item_year_df.shape

(28857, 18)

Удаляем пропуски по id_user

In [428]:
user_item_year_df = user_item_year_df[user_item_year_df["id_user"].notna() & user_item_year_df["Категория"].notna()].copy()
user_item_year_df["id_user"] = user_item_year_df["id_user"].astype("Int64")

In [429]:
user_item_year_df.shape

(28807, 18)

In [430]:
user_item_year_df.isna().sum()

Имя мастера                       0
Специализация мастера             0
Имя клиента                       0
Телефон                           0
Email                             0
Время визита                      0
Имя создателя                     0
Дата создания                     0
Статус                            0
Источник                          0
Категория услуги                  0
Название услуги                   0
Стоимость, руб                    0
Стоимость с учетом скидки, руб    0
Оплачено полностью                0
Комментарий                       0
Категория                         0
id_user                           0
dtype: int64

Удаляем неинформатиыные столбцы

In [431]:
user_item_year_df = user_item_year_df.drop(columns=['Email', 'Статус', 'Дата создания','Имя создателя',
                                           'Оплачено полностью', 'Комментарий', 'Источник', 'Телефон' ,
                                           'Стоимость с учетом скидки, руб'])
user_item_year_df.info()

<class 'pandas.DataFrame'>
Index: 28807 entries, 20 to 28856
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Имя мастера            28807 non-null  str    
 1   Специализация мастера  28807 non-null  str    
 2   Имя клиента            28807 non-null  str    
 3   Время визита           28807 non-null  str    
 4   Категория услуги       28807 non-null  str    
 5   Название услуги        28807 non-null  str    
 6   Стоимость, руб         28807 non-null  float64
 7   Категория              28807 non-null  str    
 8   id_user                28807 non-null  Int64  
dtypes: Int64(1), float64(1), str(7)
memory usage: 2.2 MB


Добавление id_item

In [432]:
'''
user_item_year_df["Название услуги_norm"] = (
    user_item_year_df["Название услуги"]
    .str.replace(" ",  "", regex=False)
    .str.replace(" ", "", regex=False)  # неразрывный пробел
    .str.strip()
    .str.lower()
)

items_dim["Название услуги или товара_norm"] = (
    items_dim["Название услуги или товара"]
    .str.replace(" ",  "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.strip()
    .str.lower()
) '''

'\nuser_item_year_df["Название услуги_norm"] = (\n    user_item_year_df["Название услуги"]\n    .str.replace(" ",  "", regex=False)\n    .str.replace(" ", "", regex=False)  # неразрывный пробел\n    .str.strip()\n    .str.lower()\n)\n\nitems_dim["Название услуги или товара_norm"] = (\n    items_dim["Название услуги или товара"]\n    .str.replace(" ",  "", regex=False)\n    .str.replace(" ", "", regex=False)\n    .str.strip()\n    .str.lower()\n) '

In [433]:
user_item_year_df['Название услуги'] = (user_item_year_df['Название услуги']
                                        .str.strip()
                                        .str.lower()
                                        .str.replace(r'\s+', ' ', regex=True))

In [434]:
items_dim['Название услуги или товара'] = (items_dim['Название услуги или товара']
                                           .str.strip()
                                           .str.lower()
                                           .str.replace(r'\s+', ' ', regex=True))

In [435]:
user_item_year_clean = pd.merge(
    user_item_year_df,
    items_dim[["Название услуги или товара", "id_item"]],
    left_on="Название услуги",
    right_on="Название услуги или товара",
    how="left"
)

In [436]:
user_item_year_clean = user_item_year_clean.drop(columns=['Название услуги или товара'])

In [437]:
user_item_year_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 28807 entries, 0 to 28806
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Имя мастера            28807 non-null  str    
 1   Специализация мастера  28807 non-null  str    
 2   Имя клиента            28807 non-null  str    
 3   Время визита           28807 non-null  str    
 4   Категория услуги       28807 non-null  str    
 5   Название услуги        28807 non-null  str    
 6   Стоимость, руб         28807 non-null  float64
 7   Категория              28807 non-null  str    
 8   id_user                28807 non-null  Int64  
 9   id_item                28807 non-null  int64  
dtypes: Int64(1), float64(1), int64(1), str(7)
memory usage: 2.2 MB


In [438]:
f"Пропусков id_item: {user_item_year_clean['id_item'].isna().sum()}"

'Пропусков id_item: 0'

### user_item_df

In [439]:
user_item_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3133 entries, 0 to 3132
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Имя	 мастера                    1777 non-null   str    
 1   Специализация мастера           1777 non-null   str    
 2   Имя клиента                     1154 non-null   str    
 3   	Телефон                        1154 non-null   float64
 4   Email                           502 non-null    str    
 5   Время визита                    1777 non-null   str    
 6   Имя	 создателя                  1777 non-null   str    
 7   Дата создания                   1777 non-null   str    
 8   Статус                          1777 non-null   str    
 9   Источник                        1777 non-null   str    
 10  Категория услуги                3096 non-null   str    
 11  Название услуги                 3096 non-null   str    
 12  Стоимость, руб                  3096 non-null

Почистить имена колонок

In [440]:
user_item_df.columns = user_item_df.columns.str.strip().str.replace(r'\t','', regex=True)

In [441]:
user_item_df.columns

Index(['Имя мастера', 'Специализация мастера', 'Имя клиента', 'Телефон',
       'Email', 'Время визита', 'Имя создателя', 'Дата создания', 'Статус',
       'Источник', 'Категория услуги', 'Название услуги', 'Стоимость, руб',
       'Стоимость с учетом скидки, руб', 'Оплачено полностью', 'Комментарий',
       'Категория'],
      dtype='str')

In [442]:
COL_MASTER = "Имя мастера"
COL_MASTER_SPEC = "Специализация мастера"
COL_CLIENT_NAME = "Имя клиента"
COL_PHONE = "Телефон"
COL_EMAIL = "Email"
COL_VISIT_TIME = "Время визита"
COL_CATEGORY = "Категория услуги"
COL_SERVICE = "Название услуги"
COL_PRICE_DISCOUNT = "Стоимость с учетом скидки, руб"

In [443]:
user_item_df[COL_PRICE_DISCOUNT] = clean_money(user_item_df[COL_PRICE_DISCOUNT])

In [444]:
user_item_df.shape

(3133, 17)

In [445]:
user_item_df = user_item_df[user_item_df[COL_SERVICE].notna()]
user_item_df = user_item_df[user_item_df[COL_SERVICE].str.strip() != ""]

In [446]:
user_item_df.shape

(3096, 17)

Фильтрация по солярию и абонимаентам 

In [447]:
mask_not_solar = user_item_df[COL_CATEGORY] != "Солярий"
mask_not_sub = user_item_df[COL_CATEGORY] != "Услуги ПО АБОНЕМЕНТАМ"
user_item_df = user_item_df[mask_not_solar & mask_not_sub].copy()

In [448]:
user_item_df.shape

(2457, 17)

In [449]:
user_item_df.isna().sum()

Имя мастера                       1349
Специализация мастера             1349
Имя клиента                       1358
Телефон                           1358
Email                             1970
Время визита                      1349
Имя создателя                     1349
Дата создания                     1349
Статус                            1349
Источник                          1349
Категория услуги                     0
Название услуги                      0
Стоимость, руб                       0
Стоимость с учетом скидки, руб       0
Оплачено полностью                1349
Комментарий                       2170
Категория                         2306
dtype: int64

In [450]:
user_item_df = user_item_df.sort_values([COL_VISIT_TIME, COL_CLIENT_NAME], ascending=False)

In [451]:
fill_cols = [col for col in user_item_df.columns if col != COL_SERVICE]
user_item_df[fill_cols] = user_item_df[fill_cols].ffill(axis=0)

In [452]:
user_item_df.isna().sum()

Имя мастера                       0
Специализация мастера             0
Имя клиента                       0
Телефон                           0
Email                             3
Время визита                      0
Имя создателя                     0
Дата создания                     0
Статус                            0
Источник                          0
Категория услуги                  0
Название услуги                   0
Стоимость, руб                    0
Стоимость с учетом скидки, руб    0
Оплачено полностью                0
Комментарий                       6
Категория                         0
dtype: int64

Ренейм (объеденяем item)

In [453]:
rename_map = {
    # брови
    "коррекция бровей/форма бровей (жен.)": "Коррекция бровей/форма бровей",
    "коррекция бровей/форма бровей (муж.)": "Коррекция бровей/форма бровей",

    # укрепление
    "укрепление акрилом": "Укрепление",
    "укрепление гелем": "Укрепление",

    # покрытие
    "укрепление цветным гелем": "Покрытие",
    "покрытие гель лаком luxio, emi, onenail (руки)": "Покрытие",
    "покрытие стойким лаком (ноги)": "Покрытие",
    "покрытие база + топ (бесцветные)": "Покрытие",
    "покрытие гель лаком со светоотражающим эффектом (руки)": "Покрытие",
    "покрытие стойким лаком (руки)": "Покрытие",
    "покрытие гель лаком luxio, emi, onenail (ноги)": "Покрытие",
    "покрытие гель лаком emi, onenail (руки)": "Покрытие",
    "покрытие гель лаком со светоотражающим эффектом (ноги)": "Покрытие",
    "покрытие гель лаком luxio, beautix, e.mi, onenail (руки) 1 ноготь": "Покрытие",
    "покрытие гель лаком с эффектом \"кошачий глаз\" (руки)": "Покрытие",
    "покрытие гель лаком luxio, beautix, onenail френч (руки)": "Покрытие",

    # снятие
    "снятие лака (руки)": "Снятие",
    "снятие гель-лака (1 ноготь)": "Снятие",
    "снятие гель-лака": "Снятие",
    "снятие стойкого лака (руки)": "Снятие",
    "снятие лака (ноги)": "Снятие",
    "снятие укрепления гелем / нарощенных ногтей (10 ногтей)": "Снятие",
}

In [454]:
user_item_df[COL_SERVICE] = user_item_df[COL_SERVICE].str.strip().str.lower().replace(rename_map)

In [455]:
user_item_df.shape

(2457, 17)

Добавляем id_user

In [456]:
user_item_df[COL_PHONE] = user_item_df[COL_PHONE].astype(str).str.strip()

In [457]:
user_item_df["Телефон"] = user_item_df["Телефон"].apply(clean_phone)

In [458]:
user_item_df = user_item_df.merge(
    users_df[["id_user", "Телефон"]],
    on="Телефон",
    how="left"
)

In [459]:
user_item_df.isna().sum()

Имя мастера                       0
Специализация мастера             0
Имя клиента                       0
Телефон                           0
Email                             3
Время визита                      0
Имя создателя                     0
Дата создания                     0
Статус                            0
Источник                          0
Категория услуги                  0
Название услуги                   0
Стоимость, руб                    0
Стоимость с учетом скидки, руб    0
Оплачено полностью                0
Комментарий                       6
Категория                         0
id_user                           2
dtype: int64

Удаляем пропуски по id_user

In [460]:
user_item_df = user_item_df[user_item_df["id_user"].notna() & user_item_df["Категория"].notna()].copy()
user_item_df["id_user"] = user_item_df["id_user"].astype("Int64")

In [461]:
user_item_df['id_user'].isna().sum()

0

Удаляем не информатиыные столбцы

In [462]:
user_item_df = user_item_df.drop(columns=['Email', 'Статус', 'Дата создания','Имя создателя',
                                           'Оплачено полностью', 'Комментарий', 'Источник', 'Телефон' ,
                                           'Стоимость с учетом скидки, руб'])
user_item_df.info()

<class 'pandas.DataFrame'>
Index: 2455 entries, 0 to 2456
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Имя мастера            2455 non-null   str    
 1   Специализация мастера  2455 non-null   str    
 2   Имя клиента            2455 non-null   str    
 3   Время визита           2455 non-null   str    
 4   Категория услуги       2455 non-null   str    
 5   Название услуги        2455 non-null   str    
 6   Стоимость, руб         2455 non-null   float64
 7   Категория              2455 non-null   str    
 8   id_user                2455 non-null   Int64  
dtypes: Int64(1), float64(1), str(7)
memory usage: 194.2 KB


Добавляем id_item

In [463]:
user_item_df['Название услуги'] = (user_item_df['Название услуги']
                                        .str.strip()
                                        .str.lower()
                                        .str.replace(r'\s+', ' ', regex=True))

In [464]:
user_item_clean = pd.merge(
    user_item_df,
    items_dim[["Название услуги или товара", "id_item"]],
    left_on="Название услуги",
    right_on="Название услуги или товара",
    how="left"
)

In [465]:
user_item_clean = user_item_clean.drop(columns=['Название услуги или товара'])

In [466]:
user_item_clean.isna().sum()

Имя мастера              0
Специализация мастера    0
Имя клиента              0
Время визита             0
Категория услуги         0
Название услуги          0
Стоимость, руб           0
Категория                0
id_user                  0
id_item                  0
dtype: int64

## Сортировка таблиц по id

In [467]:
# 1. users_clean по user_id
users_clean = users_clean.sort_values('id_user').reset_index(drop=True)

# 2. items_clean по item_id
items_clean = items_clean.sort_values('id_item').reset_index(drop=True)

# 3. user_items_final по user_id
user_item_final = user_items_final.sort_values('user_id').reset_index(drop=True)

NameError: name 'items_clean' is not defined

## Выгрузка таблиц

In [ ]:

''' 
users_clean.to_csv('users_clean_final.csv', index=False, encoding='utf-8-sig')
items_clean.to_csv('items_clean_final.csv', index=False, encoding='utf-8-sig')
user_item_final.to_csv('user_items_final.csv', index=False, encoding='utf-8-sig')
'''